# Ablation study 2 - Set 1 - one part removed, the fusion variants, and an added consistency loss

**Final model = M9 with plain concatenation** (code `S1T1M1P1`): the M9 architecture where the adaptive CNN/Transformer gate is replaced by plain concatenation. Every ablation below changes one thing, or removes some parts, of this model. The notebook trains each configuration on **10 random splits** (seeds 0-9, 720/80/200 clips, the same splits as the main study), 40 epochs, early stopping on validation Mean(6) SRCC, and compares it with the final model.

## How to run (Kaggle)
1. Add the same two datasets as for the earlier notebooks (colonoscopy videos + MOS file). Turn **GPU on** and **Internet on** (the pretrained ResNet-50 is downloaded once).
2. **Run all cells in order.** Feature extraction takes about 10-15 minutes the first time, then training starts. The run is **resumable**: if the session stops, run all cells again and it continues from `runs.csv`.
3. When it finishes, download **`cvstqa_ablation2_set1_results.zip`** from the output panel and send it back. Do not rename it.

## What this notebook runs (random split, 10 seeds each)
Row 1 is the **final model**, repeated in every set only as a shared reference so the sets can be checked against each other. Rows 2 onward are the ablations that belong to this set only: no other set runs them.

| # | Ablation (plain words) | Code | What changes |
|---|---|---|---|
| 1 | Final model (all parts on) | `S1T1M1P1` | M9 with plain concatenation fusion, all four parts on |
| 2 | Final model without spatial attention | `S0T1M1P1` | Kept: temporal attention, multi-stage features, photometric descriptors. Removed: spatial attention. |
| 3 | Final model without temporal attention | `S1T0M1P1` | Kept: spatial attention, multi-stage features, photometric descriptors. Removed: temporal attention. |
| 4 | Final model without multi-stage features | `S1T1M0P1` | Kept: spatial attention, temporal attention, photometric descriptors. Removed: multi-stage features. |
| 5 | Final model without photometric descriptors | `S1T1M1P0` | Kept: spatial attention, temporal attention, multi-stage features. Removed: photometric descriptors. |
| 6 | Adaptive gate restored | `A8_gate_restored` | The adaptive CNN/Transformer gate instead of plain concatenation (this is the previous M9) |
| 7 | CNN branch only | `A8_cnn_only` | Spatial Transformer branch removed |
| 8 | Transformer branch only | `A8_transformer_only` | Plain CNN tokens dropped |
| 9 | Added temporal-consistency loss, weight 0.05 | `A9_consistency_0.05` | A consistency loss that penalises abrupt frame-to-frame quality changes is added (not part of the final model); 0.05 is the manual's value |
| 10 | Added temporal-consistency loss, weight 0.5 | `A9_consistency_0.5` | The same loss with weight 0.5 |

Letters in the code: **S** spatial attention, **T** temporal attention, **M** multi-stage ResNet features, **P** photometric descriptors; 1 = present, 0 = removed.

## Patient-independent split (Protocol B), 10 seeds
The final model only (`S1T1M1P1`).

Approximate time on a Kaggle T4/P100: 10 configurations x 10 seeds x about 1 minute.

The tables and figures at the end (cells 8-9) cover only the configurations in this set. The full comparison across all four sets is made after the four zips are merged.

In [ ]:
# ============ CELL 1 — imports & configuration (ablation study 2: the FINAL model = M9 with plain concatenation instead of the adaptive gate) ============
import os, gc, json, math, random, time, copy, re, warnings, shutil, zipfile, platform, itertools, sys
from pathlib import Path
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from scipy.stats import pearsonr, spearmanr, kendalltau, wilcoxon
from scipy.optimize import curve_fit
import matplotlib
if "ipykernel" not in sys.modules: matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
cv2.setNumThreads(1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU")
if DEVICE.type == "cpu": print("WARNING: no GPU. Turn on a GPU (Kaggle: Settings -> Accelerator) and Internet (needed once for the pretrained ResNet-50).")

# ---------------- paths ----------------
INPUT = Path(os.environ.get("CVSTQA_INPUT", "/kaggle/input"))
WORK  = Path(os.environ.get("CVSTQA_WORK", "/kaggle/working")); WORK.mkdir(parents=True, exist_ok=True)
CACHE_MAIN = Path(os.environ.get("CVSTQA_CACHE_MAIN", "/tmp/cvstqa_cache"))          # cache of the M0-M13 notebook: reused if it is complete (same session)
CACHE = Path(os.environ.get("CVSTQA_CACHE", "/tmp/cvstqa_cache_ablation")); CACHE.mkdir(parents=True, exist_ok=True)   # otherwise the three needed features are extracted here (~10-15 min, ResNet-50 + photometric only)
SET_ID = os.environ.get("CVSTQA_SET", "1")                    # which of the four teammate sets this notebook is (1-4); only used in the output names
OUT   = WORK / f"cvstqa_ablation2_set{SET_ID}_results"
for sub in ["tables", "figures", "preds"]: (OUT / sub).mkdir(parents=True, exist_ok=True)
PRED_DIR, RUNS_CSV = OUT / "preds", OUT / "runs.csv"

# ---------------- protocol (same as the main study) ----------------
QUICK_TEST = os.environ.get("CVSTQA_SMOKE", "0") == "1"      # keep False for the real run
SEEDS      = list(range(10))          # 10 iterations; seed s -> split s (the SAME splits as the M0-M13 notebook)
EPOCHS, BATCH = 40, 32                # 720 train clips / 32 = 23 steps per epoch
EARLY_STOP, PATIENCE = True, 10
TEST_FRAC, VAL_FRAC = 0.20, 0.10
PRETRAINED, RANDOM_INIT = True, False
PATIENT_REGEX = r"^(\d+)-"            # used ONLY for the patient-independent Protocol B. CHECK that it really identifies the patient/exam (the dataset has ~40 exams): if the printout in Cell 2 shows far more groups than that, supply the true mapping (see Cell 2) before trusting Protocol B.
ATTRS = ["TSV", "B", "SR", "S", "U"]; TARGETS = ATTRS + ["O"]
SCORE_NAME = {"TSV": "TSV", "B": "B", "SR": "SR", "S": "S", "U": "U", "O": "Overall", "mean": "Mean(6)"}

# ---------------- what to run ----------------
FACTORS_ACTIVE = ["S", "T", "M", "P"]  # factorial ablation of the FINAL model (M9 with plain concatenation fusion; every cell keeps concatenation): S spatial attention | T temporal attention | M multi-stage features (layer2+3) | P photometric descriptors.
RUN_FACTORIAL   = True    # all 2^k on/off combinations of FACTORS_ACTIVE (k=4: 16 cells; S1T1M1P1 is the final model)
RUN_EXTRAS      = True    # the manual's other ablations (pooling variants, shared beta, T=8, patch grid, lambda, fusion) + controls
RUN_PROTOCOL_B  = True    # patient-independent split for a few key models
ONLY   = ['S1T1M1P1', 'S0T1M1P1', 'S1T0M1P1', 'S1T1M0P1', 'S1T1M1P0', 'A8_gate_restored', 'A8_cnn_only', 'A8_transformer_only', 'A9_consistency_0.05', 'A9_consistency_0.5']   # SET 1: the configurations this person runs (random split)
ONLY_B = ['S1T1M1P1']   # SET 1: Protocol B models (patient-independent split)
if QUICK_TEST:
    SEEDS, EPOCHS, TEST_FRAC, VAL_FRAC = [0, 1], 2, 0.30, 0.30
    PRETRAINED, RANDOM_INIT = False, True

# ---------------- fixed hyper-parameters (shared by every configuration) ----------------
HP = dict(D=256, DROPOUT=0.2, HEADS=4, SP_LAYERS=1, T_LAYERS=2, LR=5e-4, WD=1e-2, WARMUP=2, GRAD_CLIP=1.0,
          LAMBDA_RANK=0.1, RANK_MARGIN=0.05, LAMBDA_TEMP=0.5, TEMP_TAU=0.1)
NUM_FRAMES, IMAGE_SIZE, NUM_PATCHES = 16, 224, 49
CNN_DIM, MS_DIM, PH_DIM = 2048, 1536, 16

DARNET = {"TSV": (.910, .924, .744, .252), "B": (.798, .821, .607, .459), "SR": (.841, .855, .654, .343),
          "S": (.928, .933, .768, .276), "U": (.906, .914, .737, .283), "O": (.940, .943, .790, .332)}
DARNET["mean"] = tuple(round(float(np.mean([DARNET[t][i] for t in TARGETS])), 3) for i in range(4))

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
print(f"{len(SEEDS)} seeds | {EPOCHS} epochs | batch {BATCH} | factors {FACTORS_ACTIVE} | factorial {RUN_FACTORIAL} extras {RUN_EXTRAS} protocol-B {RUN_PROTOCOL_B} | quick-test {QUICK_TEST}")


In [ ]:
# ============ CELL 2 — locate inputs & parse MOS (identical to the frozen-feature notebook) ============
print("Top-level input folders:")
for p in sorted(INPUT.iterdir()):
    print("  ", p)
    for q in sorted(p.iterdir())[:6]:
        print("      ", q.name)

# optional: show what the split dataset contains (not used for training here)
for p in INPUT.rglob("*"):
    if p.is_dir() and "split" in p.name.lower():
        print("\nSplit-like folder:", p)
        for q in sorted(p.rglob("*"))[:20]:
            if q.is_file(): print("    ", q.relative_to(p), f"({q.stat().st_size/1e3:.1f} KB)")

mos_file = next(INPUT.rglob("MOS.txt"))
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
all_videos = sorted(p for p in INPUT.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
print(f"\nMOS.txt: {mos_file}\nVideos found: {len(all_videos)}")

vmap = {}
for p in all_videos:
    vmap.setdefault(p.name.lower(), str(p)); vmap.setdefault(p.stem.lower(), str(p))

mos_df = pd.read_csv(mos_file, sep=None, engine="python", header=None,
                     names=["video_id"] + TARGETS, skipinitialspace=True)
mos_df["video_id"] = mos_df["video_id"].astype(str).str.strip()
for c in TARGETS:
    mos_df[c] = pd.to_numeric(mos_df[c], errors="coerce")
mos_df = mos_df.dropna(subset=TARGETS).reset_index(drop=True)          # drops a header row if present
mos_df["video_path"] = mos_df["video_id"].str.lower().map(vmap)
if mos_df["video_path"].isna().any():
    mos_df["video_path"] = mos_df["video_id"].map(lambda s: vmap.get(Path(s).stem.lower()))
assert mos_df["video_path"].notna().all(), mos_df[mos_df["video_path"].isna()].head()
print("Labelled & matched clips:", len(mos_df))
print(mos_df[TARGETS].describe().loc[["min", "max", "mean"]].round(3))

mos_df["patient"] = mos_df["video_id"].map(lambda s: (re.match(PATIENT_REGEX, s) or [None, s])[1])
print("Unique patient/group prefixes:", mos_df["patient"].nunique(), "(expected ~40 exams - if it is much larger, the prefix is NOT the patient: Protocol B would still leak. Fix PATIENT_REGEX in Cell 1, or load a real mapping here, e.g. mos_df['patient'] = mos_df['video_id'].map(dict(pd.read_csv('patients.csv').values)))")
Y = mos_df[TARGETS].values.astype(np.float32)          # [N, 6] raw MOS
N_VIDEOS = len(mos_df)
print(f"\nClips used: {N_VIDEOS} (the full dataset has 1,000)")
if N_VIDEOS != 1000 and not QUICK_TEST: print("WARNING: expected 1,000 labelled clips - check that both the video dataset and the MOS dataset are attached.")


In [ ]:
# ============ CELL 3 — features needed by M9 only: ResNet-50 layer4 (l4), layer2+3 (ms), photometric descriptors (ph). Reuses the cache of the M0-M13 notebook if it is complete; otherwise extracts (one pass, resumable) ============
def photometric(fr):
    g = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    hsv = cv2.cvtColor(fr, cv2.COLOR_RGB2HSV); S = hsv[..., 1].astype(np.float32) / 255.0; V = hsv[..., 2].astype(np.float32) / 255.0
    R, Gc, Bc = [fr[..., i].astype(np.float32) for i in range(3)]
    rg, yb = R - Gc, 0.5 * (R + Gc) - Bc
    colorful = np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2)
    lap = cv2.Laplacian(g, cv2.CV_32F); gx = cv2.Sobel(g, cv2.CV_32F, 1, 0); gy = cv2.Sobel(g, cv2.CV_32F, 0, 1)
    edges = cv2.Canny((g * 255).astype(np.uint8), 50, 150).mean() / 255.0
    noise = np.median(np.abs(g - cv2.GaussianBlur(g, (3, 3), 0))) / 0.6745
    hist = np.histogram(g, bins=32, range=(0, 1))[0].astype(np.float32); hist /= hist.sum() + 1e-8
    p5, p95 = np.percentile(g, [5, 95])
    return np.array([g.mean(), g.std(), p5, p95, (g < 0.1).mean(), (g > 0.9).mean(), ((V > 0.94) & (S < 0.25)).mean(),
                     S.mean(), S.std(), colorful / 100.0, np.log1p(lap.var() * 1e3), np.sqrt(gx ** 2 + gy ** 2).mean(), edges, noise,
                     (R / (Gc + 1.0)).mean(), -(hist * np.log(hist + 1e-8)).sum()], np.float32)

SHAPES = {"l4": ((N_VIDEOS, NUM_FRAMES, NUM_PATCHES, CNN_DIM), np.float16), "ms": ((N_VIDEOS, NUM_FRAMES, NUM_PATCHES, MS_DIM), np.float16), "ph": ((N_VIDEOS, NUM_FRAMES, PH_DIM), np.float32)}
FEATS = list(SHAPES)
def _complete(d, need_done_file="done.npy"):
    try:
        if not (d / need_done_file).exists() or not np.load(d / need_done_file).all(): return False
        if json.load(open(d / "cache_meta.json")).get("n") != N_VIDEOS: return False
        return all((d / f"{k}.npy").exists() and tuple(np.load(d / f"{k}.npy", mmap_mode="r").shape) == SHAPES[k][0] for k in FEATS)
    except Exception: return False

class VideoLite(Dataset):
    def __init__(self, paths, order): self.paths, self.order = paths, order
    def __len__(self): return len(self.order)
    def _rd(self, cap, last):
        ok, fr = cap.read()
        if ok: return cv2.resize(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB), (IMAGE_SIZE, IMAGE_SIZE))
        return last if last is not None else np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), np.uint8)
    def __getitem__(self, k):
        i = int(self.order[k]); cap = cv2.VideoCapture(self.paths[i]); n = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
        frames, last = [], None
        for f in np.linspace(0, n - 1, NUM_FRAMES).astype(int):                                  # 16 uniformly sampled frames (identical to the main notebook)
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(f)); last = self._rd(cap, last); frames.append(last)
        cap.release()
        return torch.from_numpy(np.stack(frames)), torch.from_numpy(np.stack([photometric(f) for f in frames])), i

def resnet_stages(m, x):
    x = m.maxpool(m.relu(m.bn1(m.conv1(x)))); x = m.layer1(x); f2 = m.layer2(x); f3 = m.layer3(f2); f4 = m.layer4(f3); return f2, f3, f4

CACHE_USE = CACHE_MAIN if (not RANDOM_INIT and _complete(CACHE_MAIN)) else CACHE
if CACHE_USE == CACHE_MAIN: print(f"Reusing the complete feature cache of the M0-M13 notebook: {CACHE_MAIN}")
else:
    sig = {"feats": FEATS, "n": N_VIDEOS, "random_init": RANDOM_INIT, "ids": list(mos_df["video_id"][:5])}
    META, DONE = CACHE / "cache_meta.json", CACHE / "done.npy"
    fresh = not (META.exists() and json.load(open(META)) == sig and DONE.exists() and all((CACHE / f"{k}.npy").exists() for k in FEATS))
    mm = {k: np.lib.format.open_memmap(CACHE / f"{k}.npy", mode=("w+" if fresh else "r+"), dtype=SHAPES[k][1], shape=SHAPES[k][0]) for k in FEATS}
    done = np.zeros(N_VIDEOS, bool) if fresh else np.load(DONE); json.dump(sig, open(META, "w"))
    todo = np.where(~done)[0]; print(f"Extracting into {CACHE} | already cached {int(done.sum())}/{N_VIDEOS} | to do {len(todo)}")
    if len(todo):
        # pretrained ImageNet weights: from an attached file (works with Internet OFF) or downloaded (needs Internet ON). File name: resnet50-11ad3fb6.pth (= torchvision ResNet50_Weights.DEFAULT)
        _wf = next(iter(INPUT.rglob("resnet50-11ad3fb6.pth")), None) if INPUT.exists() else None
        if _wf is not None and PRETRAINED:
            rn = models.resnet50(weights=None); rn.load_state_dict(torch.load(_wf, map_location="cpu")); print("  loaded ResNet-50 weights from", _wf)
        else:
            try: rn = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if PRETRAINED else None)
            except Exception as e: raise SystemExit("Cannot download the ResNet-50 weights (Internet is OFF). Fix: Notebook settings -> Internet ON (the Kaggle account must be phone-verified), OR attach a dataset that contains resnet50-11ad3fb6.pth and re-run this cell. (" + type(e).__name__ + ")")
        rn = rn.to(DEVICE).eval()
        imean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1); istd = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
        dl = DataLoader(VideoLite(mos_df["video_path"].tolist(), todo), batch_size=4, num_workers=0 if QUICK_TEST else 2, pin_memory=(DEVICE.type == "cuda"))
        amp = dict(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda")); t0 = time.time()
        def to_seq(t, b, d): return t.flatten(2).transpose(1, 2).reshape(b, NUM_FRAMES, NUM_PATCHES, d)
        with torch.inference_mode():
            for bi, (fr, ph, ids) in enumerate(dl):
                b, ii = fr.shape[0], ids.numpy()
                x = fr.to(DEVICE).view(b * NUM_FRAMES, IMAGE_SIZE, IMAGE_SIZE, 3).permute(0, 3, 1, 2).float() / 255.0
                with torch.autocast(**amp): f2, f3, f4 = resnet_stages(rn, (x - imean) / istd)
                mm["l4"][ii] = to_seq(f4.float(), b, CNN_DIM).half().cpu().numpy()
                mm["ms"][ii] = to_seq(torch.cat([F.adaptive_avg_pool2d(f2.float(), 7), F.adaptive_avg_pool2d(f3.float(), 7)], 1), b, MS_DIM).half().cpu().numpy()
                mm["ph"][ii] = ph.numpy(); done[ii] = True
                if bi % 25 == 0:
                    for v in mm.values(): v.flush()
                    np.save(DONE, done); print(f"  {int(done.sum())}/{N_VIDEOS} cached | {time.time()-t0:.0f}s")
        for v in mm.values(): v.flush()
        np.save(DONE, done); del rn
    assert done.all(), "Some videos failed to cache"
    del mm; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
print("Feature cache ready:", CACHE_USE)


In [ ]:
# ============ CELL 4 — load the cached features (GPU if they fit) ============
STORE = {}
def _fits(nbytes):
    if DEVICE.type != "cuda": return False
    return nbytes < torch.cuda.mem_get_info()[0] - 5.5e9          # keep ~5.5 GB headroom for training
def load_big(name):
    arr = np.load(CACHE_USE / f"{name}.npy", mmap_mode="r"); nb = arr.size * arr.dtype.itemsize
    if _fits(nb):
        t = torch.empty(arr.shape, dtype=torch.float16, device=DEVICE)
        for i in range(0, len(arr), 50): t[i:i + 50] = torch.from_numpy(np.ascontiguousarray(arr[i:i + 50])).to(DEVICE)
        where = "GPU"
    else: t = torch.from_numpy(np.ascontiguousarray(arr)); where = "CPU RAM"
    print(f"  {name:3s} {tuple(arr.shape)} {nb/1e9:.2f} GB -> {where}"); return t
for k in ["l4", "ms"]:
    STORE[f"p_{k}"] = load_big(k)
    STORE[f"g_{k}"] = torch.cat([STORE[f"p_{k}"][i:i + 50].to(DEVICE).float().mean(2).half() for i in range(0, len(STORE[f"p_{k}"]), 50)])     # per-frame global feature = mean of the 49 patches
STORE["ph"] = torch.from_numpy(np.load(CACHE_USE / "ph.npy")).to(DEVICE)
Y = mos_df[TARGETS].values.astype(np.float32)
PH_PERM = torch.from_numpy(np.random.default_rng(12345).permutation(N_VIDEOS))      # fixed permutation for the "photometric shuffled" control
print("loaded:", {k: tuple(v.shape) for k, v in STORE.items()})
def take(name, idx):
    t = STORE[name]; return t[idx.to(t.device)].to(DEVICE, non_blocking=True).float()


In [ ]:
# ============ CELL 5 — the FINAL model (M9 with plain concatenation fusion) as a configurable network, and every ablation configuration ============
# M9 = patch tokens (ResNet-50 layer4 + layer2/3) -> per-patch attribute quality q and spatial criticality alpha -> frame quality Q_t
#      -> temporal Transformer -> attribute-specific temporal criticality beta -> video attribute scores (+ global residual) -> learned Overall head; photometric descriptors added to the frame context.
BASE = dict(use_patches=True, temporal="transformer", spatial_pool="attn", temporal_pool="attn", temp_loss=False, global_resid=True,
            ms=True, ph=True, grid=7, stride=1, fusion="concat", lam_rank=None, lam_temp=None, rank_overall_only=False, ph_shuffle=False)
def V(**kw):
    d = dict(BASE); d.update(kw); return d

FACTOR_NAME = {"S": "spatial attention (alpha)", "T": "temporal attention (beta)", "M": "multi-stage features (layer2+3)", "P": "photometric descriptors", "C": "temporal-consistency loss"}
def cell_cfg(bits):                       # bits: dict factor -> 0/1
    return V(spatial_pool="attn" if bits.get("S", 1) else "avg", temporal_pool="attn" if bits.get("T", 1) else "mean",
             ms=bool(bits.get("M", 1)), ph=bool(bits.get("P", 1)), temp_loss=bool(bits.get("C", 0)))
def cell_name(bits): return "".join(f"{f}{int(bits[f])}" for f in FACTORS_ACTIVE)
FULL_BITS = {f: (0 if f == "C" else 1) for f in FACTORS_ACTIVE}; FULL = cell_name(FULL_BITS)
ALIAS = {"S1T1M1P1": "FINAL = M9 + plain concatenation"}

VARIANTS, GROUP, DESC = {}, {}, {}
def add(name, cfg, group, desc): VARIANTS[name], GROUP[name], DESC[name] = cfg, group, desc
for combo in itertools.product([1, 0], repeat=len(FACTORS_ACTIVE)):                       # 2^k cells, full model first
    bits = dict(zip(FACTORS_ACTIVE, combo)); nm = cell_name(bits)
    add(nm, cell_cfg(bits), "factorial", ALIAS.get(nm, "") + " | " + ", ".join(f"{'+' if b else '-'}{f}" for f, b in bits.items()))
EXTRA = {                                   # the manual's other ablations (sections 8-10, 20, 29-33) and controls; each differs from the full model in ONE thing
    "A2_spatial_max":       (V(spatial_pool="max"),            "manual S31: spatial pooling = max over patches (instead of learned attention)"),
    "A3_temporal_max":      (V(temporal_pool="max"),           "manual S32: temporal pooling = max over frames"),
    "A4_shared_beta":       (V(temporal_pool="attn_shared"),   "manual S33: ONE temporal attention shared by all five attributes"),
    "A5_T8":                (V(stride=2),                      "manual S30: T = 8 frames (every 2nd cached frame); T = 24/32 need a re-extraction"),
    "A6_grid4":             (V(grid=4),                        "manual S29 (proxy): 7x7 grid average-pooled to 4x4 (larger patches)"),
    "A6_grid3":             (V(grid=3),                        "manual S29 (proxy): 3x3 grid"),
    "A6_grid2":             (V(grid=2),                        "manual S29 (proxy): 2x2 grid"),
    "A8_gate_restored":     (V(fusion="gate"),                 "manual S10: the adaptive CNN-Transformer gate restored instead of plain concatenation (= the previous M9)"),
    "A8_cnn_only":          (V(fusion="cnn"),                  "manual S9-10: no spatial contextual Transformer branch (CNN tokens only)"),
    "A8_transformer_only":  (V(fusion="trans"),                "manual S9-10: Transformer branch only"),
    "A9_rank0":             (V(lam_rank=0.0),                  "manual S19-20: no ranking loss"),
    "A9_rank0.05":          (V(lam_rank=0.05),                 "manual S20: lambda_r = 0.05"),
    "A9_rank0.2":           (V(lam_rank=0.2),                  "manual S20: lambda_r = 0.2"),
    "A9_rank0.5":           (V(lam_rank=0.5),                  "manual S20: lambda_r = 0.5"),
    "A9_rank_overall_only": (V(rank_overall_only=True),        "manual S19: ranking loss on the Overall score only (default: all six scores)"),
    "A9_consistency_0.05":  (V(temp_loss=True, lam_temp=0.05), "manual S18/S20: temporal-consistency loss with the manual's lambda_t = 0.05"),
    "A9_consistency_0.5":   (V(temp_loss=True, lam_temp=0.5),  "temporal-consistency loss with lambda_t = 0.5 (the value used for M6)"),
    "X_no_temporal_enc":    (V(temporal="none"),               "no temporal Transformer (beta then acts on frame features directly)"),
    "CTRL_ph_shuffled":     (V(ph_shuffle=True),               "control: photometric vectors shuffled across clips (same parameters, no real information)"),
}
for k, (c, d) in EXTRA.items(): add(k, c, "extra", d)
PROTO_B_MODELS = [FULL]      # patient-independent split: only the final model
PROTO_B_MODELS = list(dict.fromkeys(PROTO_B_MODELS))

def runs_for(protocol):
    names = []
    if protocol == "A":
        if RUN_FACTORIAL: names += [n for n in VARIANTS if GROUP[n] == "factorial"]
        if RUN_EXTRAS: names += [n for n in VARIANTS if GROUP[n] == "extra"]
    elif RUN_PROTOCOL_B: names = [n for n in PROTO_B_MODELS if n in VARIANTS]
    keep = ONLY if protocol == "A" else ONLY_B
    if keep is not None: names = [n for n in names if n in keep]
    return list(dict.fromkeys(names))

def mlp(i, h, o, p): return nn.Sequential(nn.Linear(i, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, o))
def tf_encoder(d, layers, p, heads):
    return nn.TransformerEncoder(nn.TransformerEncoderLayer(d, heads, 2 * d, p, activation="gelu", batch_first=True, norm_first=True), layers, enable_nested_tensor=False)

class Net(nn.Module):
    def __init__(self, cfg, hp, norm=None):
        super().__init__(); d, p, H = hp["D"], hp["DROPOUT"], hp["HEADS"]; self.cfg, self.hp = cfg, hp
        T, N = NUM_FRAMES // cfg["stride"], cfg["grid"] ** 2
        self.register_buffer("ph_mu", torch.zeros(PH_DIM)); self.register_buffer("ph_sd", torch.ones(PH_DIM))
        if norm is not None:
            for k, v in norm.items(): getattr(self, k).copy_(torch.as_tensor(v))
        gdim = CNN_DIM + (MS_DIM if cfg["ms"] else 0)
        self.gproj = nn.Sequential(nn.Dropout(p), nn.Linear(gdim, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p))
        if cfg["ph"]: self.phf, self.phc = mlp(PH_DIM, d, d, p), mlp(2 * PH_DIM, d, d, p)
        if cfg["temporal"] == "transformer":
            self.tpos = nn.Parameter(torch.zeros(1, T, d)); nn.init.trunc_normal_(self.tpos, std=0.02)
            self.tenc, self.tnorm = tf_encoder(d, hp["T_LAYERS"], p, H), nn.LayerNorm(d)
        if not cfg["use_patches"]:
            self.head = mlp(d, d, len(TARGETS), p); return
        self.pproj = nn.Sequential(nn.Dropout(p), nn.Linear(gdim, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p))
        self.ppos = nn.Parameter(torch.zeros(1, N, d)); nn.init.trunc_normal_(self.ppos, std=0.02)
        self.sp_enc = tf_encoder(d, hp["SP_LAYERS"], p, H)
        if cfg["fusion"] == "gate": self.gate = nn.Sequential(nn.Linear(2 * d, d), nn.Sigmoid())
        elif cfg["fusion"] == "concat": self.catf = nn.Sequential(nn.Linear(2 * d, d), nn.LayerNorm(d), nn.GELU())
        self.qh = nn.ModuleList([mlp(d, d // 2, 1, p) for _ in ATTRS])                              # local quality per attribute
        if cfg["spatial_pool"] == "attn": self.sh = nn.ModuleList([mlp(d, d // 2, 1, p) for _ in ATTRS])
        self.fuse = nn.Sequential(nn.Linear(d + len(ATTRS), d), nn.LayerNorm(d), nn.GELU())
        if cfg["temporal_pool"] == "attn": self.th = nn.ModuleList([mlp(d, d // 4, 1, p) for _ in ATTRS])
        elif cfg["temporal_pool"] == "attn_shared": self.th = nn.ModuleList([mlp(d, d // 4, 1, p)])
        if cfg["global_resid"]:
            self.gres = nn.Linear(d, len(ATTRS)); nn.init.zeros_(self.gres.weight); nn.init.zeros_(self.gres.bias)
        self.oh = mlp(len(ATTRS) + d, d, 1, p)

    def temporal(self, x): return self.tnorm(self.tenc(x + self.tpos)) if self.cfg["temporal"] == "transformer" else x

    def forward(self, B):
        cfg = self.cfg
        G = B["g_l4"] if not cfg["ms"] else torch.cat([B["g_l4"], B["g_ms"]], -1)
        g = self.gproj(G); pc = None
        if cfg["ph"]:
            ph = (B["ph"] - self.ph_mu) / self.ph_sd; g = g + self.phf(ph); pc = self.phc(torch.cat([ph.mean(1), ph.std(1)], -1))
        if not cfg["use_patches"]: return dict(pred=self.head(self.temporal(g).mean(1)), Q=None, alpha=None, beta=None, G=B["g_l4"])
        b, T = g.shape[:2]; N = cfg["grid"] ** 2
        P = B["p_l4"] if not cfg["ms"] else torch.cat([B["p_l4"], B["p_ms"]], -1)
        x = self.pproj(P.reshape(b * T, N, -1)); c = self.sp_enc(x + self.ppos)
        if cfg["fusion"] == "gate": gm = self.gate(torch.cat([x, c], -1)); f = gm * x + (1 - gm) * c          # adaptive CNN/Transformer fusion
        elif cfg["fusion"] == "concat": f = self.catf(torch.cat([x, c], -1))
        elif cfg["fusion"] == "cnn": f = x
        else: f = c
        q = torch.sigmoid(torch.cat([h(f) for h in self.qh], -1))                                              # [bT,N,5] local quality fields
        alpha = None
        if cfg["spatial_pool"] == "attn": alpha = torch.cat([h(f) for h in self.sh], -1).softmax(1); Qf = (alpha * q).sum(1)
        elif cfg["spatial_pool"] == "max": Qf = q.max(1).values
        else: Qf = q.mean(1)
        Q = Qf.view(b, T, len(ATTRS)); z = self.temporal(self.fuse(torch.cat([g, Q], -1))); beta = None
        if cfg["temporal_pool"] == "attn": beta = torch.cat([h(z) for h in self.th], -1).softmax(1); ya = (beta * Q).sum(1)
        elif cfg["temporal_pool"] == "attn_shared": beta = self.th[0](z).softmax(1).expand(-1, -1, len(ATTRS)); ya = (beta * Q).sum(1)
        elif cfg["temporal_pool"] == "max": ya = Q.max(1).values
        else: ya = Q.mean(1)
        zg = z.mean(1) + (pc if pc is not None else 0)
        if cfg["global_resid"]: ya = ya + self.gres(zg)
        yo = self.oh(torch.cat([ya, zg], -1))
        return dict(pred=torch.cat([ya, yo], -1), Q=Q, alpha=None if alpha is None else alpha.view(b, T, N, len(ATTRS)), beta=beta, G=B["g_l4"])

def count_params(name): return sum(p.numel() for p in Net(VARIANTS[name], HP).parameters() if p.requires_grad)
ALL_NAMES = list(dict.fromkeys(runs_for("A") + runs_for("B")))
PARAMS = {m: count_params(m) for m in ALL_NAMES}
print(f"{len([n for n in ALL_NAMES if n in runs_for('A')])} configurations for Protocol A, {len(runs_for('B'))} for Protocol B (patient-independent), {len(SEEDS)} seeds each")
print(pd.DataFrame({m: dict(params=PARAMS[m], group=GROUP[m], what=DESC[m]) for m in ALL_NAMES}).T.to_string())


In [ ]:
# ============ CELL 6 — batches, losses, logistic mapping (reference-repo function), metrics, training, timing, splits ============
# ---------------- non-linear logistic mapping: the 4-parameter logistic of nithincbabu7/non-lin-fit-python (analysis.py), same call pattern ----------------
class NonLinearFit:
    """Re-implementation of NonLinearFit.four_param_logistic_QA from the reference repo:
       f(y) = (b1 - b2) / (1 + exp((b3 - y) / |b4|)) + b2,  fitted with scipy curve_fit(f, y=prediction, x=MOS), maxfev=10000.
       (only change: the exponent is clipped to +-60 so that np.exp cannot overflow.)"""
    def __init__(self): self.maxfev = 10000; self.params = None
    @staticmethod
    def four_param_logistic_QA(x, beta1, beta2, beta3, beta4):
        return ((beta1 - beta2) / (1 + np.exp(np.clip((beta3 - x) / (np.abs(beta4) + 1e-12), -60, 60)))) + beta2
    def fit(self, x, y, beta_init=None):          # x = target MOS, y = model prediction (same argument order as the reference code)
        if beta_init is None: beta_init = [np.min(x), np.max(x), np.mean(y), 1]
        self.params, _ = curve_fit(self.four_param_logistic_QA, y, x, p0=beta_init, maxfev=self.maxfev)
    def transform(self, y): return self.four_param_logistic_QA(y, *self.params)

def fit_mapping(mos, pred):
    """Fit the mapping on (pred -> mos). Tries the reference initial guess [min(MOS), max(MOS), mean(pred), 1] and, as a fallback, an increasing-curve guess;
    keeps the better fit, and only if it beats the identity mapping on the data it was fitted on. Returns (fit or None, ok, which_init)."""
    best = None
    for k, b0 in enumerate([[mos.min(), mos.max(), pred.mean(), 1.0], [mos.max(), mos.min(), pred.mean(), pred.std() / 4 + 1e-3]]):
        try:
            f = NonLinearFit(); f.fit(mos, pred, b0); err = np.mean((f.transform(pred) - mos) ** 2)
            if np.isfinite(err) and (best is None or err < best[0]): best = (err, f, k)
        except Exception: pass
    if best is None or best[0] > np.mean((pred - mos) ** 2): return None, False, -1
    return best[1], True, best[2]
def apply_mapping(f, pred): return pred if f is None else f.transform(pred)

def _corr(fn, a, b): return 0.0 if np.std(a) < 1e-9 or np.std(b) < 1e-9 else float(fn(a, b)[0])
def _rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def mean_srcc(y, p): return float(np.mean([_corr(spearmanr, y[:, j], p[:, j]) for j in range(y.shape[1])]))
METRICS = ["srcc", "krcc", "plcc_raw", "rmse_raw", "plcc_val", "rmse_val", "plcc_test", "rmse_test"]
def all_metrics(y_te, p_te, y_va, p_va):
    """per score: SRCC, KRCC, PLCC/RMSE raw, with the mapping fitted on VALIDATION (applied fixed to test) and fitted on TEST; the 4 logistic parameters of each fit; plus the six-score mean."""
    r = {}
    for j, t in enumerate(TARGETS):
        a, b = y_te[:, j], p_te[:, j]
        r[f"{t}_srcc"], r[f"{t}_krcc"] = _corr(spearmanr, a, b), _corr(kendalltau, a, b)
        r[f"{t}_plcc_raw"], r[f"{t}_rmse_raw"] = _corr(pearsonr, a, b), _rmse(a, b)
        for tag, (fx, fy) in {"val": (y_va[:, j], p_va[:, j]), "test": (a, b)}.items():
            f, ok, k = fit_mapping(fx, fy); m = apply_mapping(f, b)
            r[f"{t}_plcc_{tag}"], r[f"{t}_rmse_{tag}"], r[f"{t}_fitok_{tag}"], r[f"{t}_init_{tag}"] = _corr(pearsonr, a, m), _rmse(a, m), ok, k
            for q in range(4): r[f"{t}_{tag[0]}b{q+1}"] = float(f.params[q]) if f is not None else np.nan      # vb1..vb4 (val-fit), tb1..tb4 (test-fit)
    for m in METRICS: r[f"mean_{m}"] = float(np.mean([r[f"{t}_{m}"] for t in TARGETS]))
    return r
def make_split(seed):
    rng = np.random.default_rng(seed); n = N_VIDEOS; perm = rng.permutation(n)
    n_te = int(round(TEST_FRAC * n)); te, rest = perm[:n_te], perm[n_te:]
    n_va = int(round(VAL_FRAC * len(rest))); va, tr = rest[:n_va], rest[n_va:]
    assert not (set(tr) & set(te)) and not (set(va) & set(te)) and not (set(tr) & set(va))
    return np.sort(tr), np.sort(va), np.sort(te)

def make_split_group(seed):
    """Protocol B: patient-independent split. Whole groups (PATIENT_REGEX prefix) go to test, then validation, the rest is train. Sizes are approximately 200 / 80 / 720."""
    rng = np.random.default_rng(seed); pats = mos_df["patient"].values; order = rng.permutation(np.unique(pats)); n = len(pats)
    def take_groups(pool, target):
        chosen, c = [], 0
        for u in pool:
            if c >= target: break
            chosen.append(u); c += int((pats == u).sum())
        return chosen
    te_g = take_groups(order, int(round(TEST_FRAC * n))); rest_g = [u for u in order if u not in set(te_g)]
    n_rest = int((~np.isin(pats, te_g)).sum()); va_g = take_groups(rest_g, int(round(VAL_FRAC * n_rest)))
    te, va = np.where(np.isin(pats, te_g))[0], np.where(np.isin(pats, va_g))[0]; tr = np.where(~np.isin(pats, te_g + va_g))[0]
    assert not (set(tr) & set(te)) and not (set(va) & set(te)) and not (set(tr) & set(va)) and not (set(pats[tr]) & set(pats[te])) and not (set(pats[va]) & set(pats[te]))
    return np.sort(tr), np.sort(va), np.sort(te)
def split_for(protocol, seed): return make_split(seed) if protocol == "A" else make_split_group(seed)

# ---------------- batches ----------------
def pool_grid(x, g):                                  # [b,T,49,C] -> [b,T,g*g,C]  (average-pool the 7x7 grid to g x g)
    if g == 7: return x
    b, T, N, C = x.shape
    return F.adaptive_avg_pool2d(x.reshape(b * T, 7, 7, C).permute(0, 3, 1, 2), g).flatten(2).transpose(1, 2).reshape(b, T, g * g, C)
FRAME_PERM = torch.from_numpy(np.random.default_rng(777).permutation(NUM_FRAMES))
def get_batch(idx, cfg, shuffle_t=False):
    idx = torch.as_tensor(idx); s, g = cfg["stride"], cfg["grid"]
    fr = (lambda t: t[:, FRAME_PERM.to(t.device)][:, ::s]) if shuffle_t else (lambda t: t[:, ::s])
    B = {"g_l4": fr(take("g_l4", idx))}
    if cfg["ms"]: B["g_ms"] = fr(take("g_ms", idx))
    if cfg["use_patches"]:
        B["p_l4"] = pool_grid(fr(take("p_l4", idx)), g)
        if cfg["ms"]: B["p_ms"] = pool_grid(fr(take("p_ms", idx)), g)
    if cfg["ph"]: B["ph"] = fr(take("ph", PH_PERM[idx] if cfg["ph_shuffle"] else idx))
    return B

def compute_losses(out, y, cfg, hp):
    pred = out["pred"]; reg = F.mse_loss(pred, y)
    lam_r = hp["LAMBDA_RANK"] if cfg["lam_rank"] is None else cfg["lam_rank"]
    pr, yr = (pred[:, 5:], y[:, 5:]) if cfg["rank_overall_only"] else (pred, y)
    pdiff = pr[:, None] - pr[None]; tdiff = yr[:, None] - yr[None]; valid = (tdiff > 0.05).float()
    rank = (valid * F.relu(hp["RANK_MARGIN"] - pdiff)).sum() / (valid.sum() + 1e-6)
    loss = reg + lam_r * rank
    if cfg["temp_loss"] and out["Q"] is not None:
        Gn = F.normalize(out["G"], dim=-1); dist = 1 - (Gn[:, 1:] * Gn[:, :-1]).sum(-1); wt = torch.exp(-dist / hp["TEMP_TAU"]).detach()
        lam_t = hp["LAMBDA_TEMP"] if cfg["lam_temp"] is None else cfg["lam_temp"]
        loss = loss + lam_t * (wt * (out["Q"][:, 1:] - out["Q"][:, :-1]).pow(2).sum(-1)).mean()
    return loss

def _sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()
@torch.no_grad()
def predict(model, idx, lo, hi, cfg, bs=64, shuffle_t=False):
    model.eval(); ps = []
    for i in range(0, len(idx), bs): ps.append(model(get_batch(torch.as_tensor(idx[i:i + bs]), cfg, shuffle_t))["pred"].float().cpu())
    return torch.cat(ps).numpy() * (hi - lo) + lo                 # back to the original MOS scale

@torch.no_grad()
def attn_stats(model, idx, cfg, bs=64):
    """Attention-collapse monitors of the manual (S39.3, S39.4) and the attribute-specificity check (S33): normalised entropy of alpha and beta (1 = uniform, 0 = one patch/frame) and the mean pairwise L1/2 distance between the five attribute-specific beta curves."""
    model.eval(); ea, eb, bd = [], [], []
    for i in range(0, len(idx), bs):
        o = model(get_batch(torch.as_tensor(idx[i:i + bs]), cfg))
        if o["alpha"] is not None: a = o["alpha"]; ea.append((-(a * (a + 1e-9).log()).sum(2) / math.log(a.shape[2])).mean().item())
        if o["beta"] is not None:
            b_ = o["beta"]; eb.append((-(b_ * (b_ + 1e-9).log()).sum(1) / math.log(b_.shape[1])).mean().item())
            bd.append(np.mean([0.5 * (b_[:, :, p] - b_[:, :, q]).abs().sum(1).mean().item() for p, q in itertools.combinations(range(5), 2)]))
    nan = float("nan"); return dict(ent_alpha=float(np.mean(ea)) if ea else nan, ent_beta=float(np.mean(eb)) if eb else nan, beta_attr_diff=float(np.mean(bd)) if bd else nan)

def make_norm(tr):
    ph = STORE["ph"][torch.as_tensor(tr).to(STORE["ph"].device)].reshape(-1, PH_DIM); return {"ph_mu": ph.mean(0).cpu(), "ph_sd": (ph.std(0) + 1e-6).cpu()}

def train_member(cfg, hp, tr, va, seed, lo, hi, Ys, norm):
    seed_everything(seed); model = Net(cfg, hp, norm).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["LR"], weight_decay=hp["WD"]); W = hp["WARMUP"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (e + 1) / W if e < W else max(0.02, 0.5 * (1 + math.cos(math.pi * (e - W) / max(1, EPOCHS - W)))))
    best, best_state, best_ep, best_step, bad, steps, steps_ep1, ep = -9, None, 0, 0, 0, 0, 0, 0; tr_t = torch.as_tensor(tr)
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = tr_t[torch.randperm(len(tr_t))]; nb = 0
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            if len(idx) < 4: continue
            loss = compute_losses(model(get_batch(idx, cfg)), Ys[idx.to(Ys.device)], cfg, hp)
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), hp["GRAD_CLIP"]); opt.step(); nb += 1; steps += 1
        if ep == 1: steps_ep1 = nb
        sched.step(); vs = mean_srcc(Y[va], predict(model, va, lo, hi, cfg))
        if vs > best + 1e-4:
            best, best_ep, best_step, bad = vs, ep, steps, 0; best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if EARLY_STOP and bad >= PATIENCE: break
    model.load_state_dict(best_state); model.eval()
    return model, dict(best_epoch=best_ep, step_at_best=best_step, epochs_run=ep, steps_per_epoch=steps_ep1, total_steps=steps, best_val_srcc=best)

def run(name, protocol, split, seed):
    cfg, hp = VARIANTS[name], HP; tr, va, te = split
    lo, hi = Y[tr].min(0), Y[tr].max(0) + 1e-8; Ys = torch.from_numpy((Y - lo) / (hi - lo)).float().to(DEVICE)
    _sync(); t0 = time.time(); model, info = train_member(cfg, hp, tr, va, seed, lo, hi, Ys, make_norm(tr)); _sync(); t_train = time.time() - t0
    pv = predict(model, va, lo, hi, cfg)
    _sync(); t1 = time.time(); pt = predict(model, te, lo, hi, cfg); _sync(); t_test = time.time() - t1          # the test set is touched once, after training
    np.save(PRED_DIR / f"{protocol}_{name}_s{seed}_val.npy", pv); np.save(PRED_DIR / f"{protocol}_{name}_s{seed}_test.npy", pt)
    res = all_metrics(Y[te], pt, Y[va], pv); res.update(attn_stats(model, te, cfg) if cfg["use_patches"] else dict(ent_alpha=np.nan, ent_beta=np.nan, beta_attr_diff=np.nan))
    if name == FULL and protocol == "A":                                                     # does the full model use frame ORDER? (test frames shuffled; the model is not retrained)
        ps = predict(model, te, lo, hi, cfg, shuffle_t=True); res["shuf_O_srcc"] = _corr(spearmanr, Y[te][:, 5], ps[:, 5]); res["shuf_mean_srcc"] = mean_srcc(Y[te], ps)
    res.update(val_mean_srcc=mean_srcc(Y[va], pv), best_epoch=info["best_epoch"], step_at_best=info["step_at_best"], epochs_run=info["epochs_run"], steps_per_epoch=info["steps_per_epoch"],
               total_steps=info["total_steps"], train_time_s=t_train, test_time_s=t_test, test_ms_per_clip=1000 * t_test / len(te), n_params=PARAMS[name], n_train=len(tr), n_val=len(va), n_test=len(te))
    del model; gc.collect()
    return res
print("steps per epoch:", math.ceil(len(make_split(0)[0]) / BATCH), "| epochs (max):", EPOCHS, "| Protocol A split sizes (train/val/test):", [len(a) for a in make_split(0)],
      "| Protocol B:", [len(a) for a in make_split_group(0)] if RUN_PROTOCOL_B else "off")


In [ ]:
# ============ CELL 7 — RUN: every configuration x every seed (resumable; one row per run is appended to runs.csv) ============
# Protocol A = the random split used by DARNet / the main study (seed s -> split s). Protocol B = patient-independent split (key models only).
rows = pd.read_csv(RUNS_CSV).to_dict("records") if RUNS_CSV.exists() else []
done_keys = {(r["protocol"], int(r["seed"]), r["model"]) for r in rows}
plan = [(p, s, m) for p in (["A"] + (["B"] if RUN_PROTOCOL_B else [])) for s in SEEDS for m in runs_for(p)]
print(f"{len([k for k in plan if k in done_keys])} of {len(plan)} runs already done")
t_all = time.time()
def seed_table(protocol, seed):
    d = pd.DataFrame([r for r in rows if r["protocol"] == protocol and int(r["seed"]) == seed and r["model"] in set(runs_for(protocol))])
    if not len(d): return
    t = pd.DataFrame({"model": d["model"]})
    for s in TARGETS: t[f"SRCC {SCORE_NAME[s]}"] = d[f"{s}_srcc"].round(4)
    t["Mean6"] = d["mean_srcc"].round(4); t["O PLCC"] = d["O_plcc_val"].round(4); t["O KRCC"] = d["O_krcc"].round(4); t["O RMSE"] = d["O_rmse_val"].round(4); t["train s"] = d["train_time_s"].round(1); t["best ep"] = d["best_epoch"].round(0)
    print(t.to_string(index=False))
for protocol in (["A"] + (["B"] if RUN_PROTOCOL_B else [])):
    for it, seed in enumerate(SEEDS):
        split = split_for(protocol, seed); did = False
        for name in runs_for(protocol):
            if (protocol, seed, name) in done_keys: continue
            r = run(name, protocol, split, seed); did = True
            r.update(seed=seed, model=name, protocol=protocol, group=GROUP[name], epochs_cfg=EPOCHS, early_stop=EARLY_STOP)
            rows.append(r); pd.DataFrame(rows).to_csv(RUNS_CSV, index=False)
            print(f"[{protocol} | iteration {it+1}/{len(SEEDS)} | seed {seed}] {name:22s} | train {r['train_time_s']:6.1f}s | test {r['test_ms_per_clip']:.2f} ms/clip | best epoch {r['best_epoch']:.0f}/{r['epochs_run']:.0f} | "
                  f"Overall SRCC {r['O_srcc']:.4f} PLCC {r['O_plcc_val']:.4f} KRCC {r['O_krcc']:.4f} RMSE {r['O_rmse_val']:.4f} | Mean(6) SRCC {r['mean_srcc']:.4f}")
            if DEVICE.type == "cuda": torch.cuda.empty_cache()
        if did: print(f"\n----- protocol {protocol}, seed {seed}: all configurations (val-fit mapping for PLCC/RMSE) -----"); seed_table(protocol, seed); print()
print(f"Finished. This session: {(time.time() - t_all) / 3600:.2f} h | runs.csv has {len(rows)} rows")


In [ ]:
# ============ CELL 8 — tables: configurations, paired tests vs M9, factorial effects, manual ablations, attention monitors, Protocol B ============
rs_all = pd.read_csv(RUNS_CSV)
PA, PB = rs_all[rs_all["protocol"] == "A"].copy(), rs_all[rs_all["protocol"] == "B"].copy()
ONLY_SAVED, ONLY, ONLY_B_SAVED, ONLY_B = ONLY, None, ONLY_B, None          # tables always cover EVERY configuration found in runs.csv (merge the runs.csv files of all sets first)
names_A = [n for n in runs_for("A") if n in set(PA["model"])]
if FULL not in names_A: raise SystemExit(f"The full model {FULL} is not in runs.csv. Merge the runs.csv files of all four sets into one (see the README / chat) and run Cells 1, 5, 8, 9, 10 again.")
_missing = [n for n in runs_for("A") if n not in names_A]
if _missing and ONLY_SAVED is None: print("WARNING: these configurations are missing from runs.csv and are left out of the tables:", _missing)
seeds_A = sorted(set.intersection(*[set(PA.loc[PA["model"] == m, "seed"]) for m in names_A])); PA = PA[PA["seed"].isin(seeds_A) & PA["model"].isin(names_A)]
SC = TARGETS + ["mean"]; SN = SCORE_NAME; TB = OUT / "tables"
def lab(m): return f"{m} [{ALIAS[m]}]" if m in ALIAS else m
print(f"Protocol A: {len(names_A)} configurations x {len(seeds_A)} seeds {seeds_A} | Protocol B: {sorted(set(PB['model']))} x {sorted(set(PB['seed']))}")
FAC = [n for n in names_A if GROUP[n] == "factorial"]; EXT = [n for n in names_A if GROUP[n] == "extra"]; REF = [n for n in names_A if GROUP[n] == "reference"]
def col(df, m, c): return df[df["model"] == m][c]
def tbl(df, models_, st, suffix, scores=SC): return pd.DataFrame({SN[s]: [getattr(col(df, m, f"{s}_{suffix}"), st)() for m in models_] for s in scores}, index=[lab(m) for m in models_])
HEAD = [("SRCC", "srcc"), ("PLCC", "plcc_val"), ("KRCC", "krcc"), ("RMSE", "rmse_val")]
def head(df, models_, st, mets=HEAD, scores=("O", "mean")): return pd.DataFrame({f"{SN[s]} {nm}": [getattr(col(df, m, f"{s}_{sf}"), st)() for m in models_] for s in scores for nm, sf in mets}, index=[lab(m) for m in models_])
def md(num, std=None, dec=4):
    cols = list(num.columns); best = {c: (num[c].idxmin() if "RMSE" in c else num[c].idxmax()) for c in cols}
    out = ["| Configuration | " + " | ".join(cols) + " |", "|" + "|".join(["---"] * (len(cols) + 1)) + "|"]
    for i, r in num.iterrows():
        cells = [f"{r[c]:.{dec}f}" + (f" ± {std.loc[i, c]:.{dec}f}" if std is not None else "") for c in cols]
        out.append("| " + " | ".join([str(i)] + [f"**{x}**" if best[c] == i else x for x, c in zip(cells, cols)]) + " |")
    return "\n".join(out)
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty(len(p)); mx = 0
    for rank, i in enumerate(o): mx = max(mx, (len(p) - rank) * p[i]); adj[i] = min(1.0, mx)
    return adj
def wp(x, y=None):
    try: return float((wilcoxon(x) if y is None else wilcoxon(x, y)).pvalue)
    except Exception: return 1.0

R_ = [f"# Ablation study 2: the final model (M9 with plain concatenation)\nProtocol A (random split, seed s = split s, identical to the main study): **{len(seeds_A)} seeds** ({seeds_A}). PLCC/RMSE use the validation-fitted logistic mapping unless stated. Bold = best in a column. Full model = **{lab(FULL)}**.\n",
      "Factors: " + "; ".join(f"**{f}** = {FACTOR_NAME[f]}" for f in FACTORS_ACTIVE) + ". A cell name lists each factor with 1 = present, 0 = removed (S1T1M1P1 is the FINAL model; every cell keeps plain concatenation fusion).\n"]
groups = [("Factorial cells (all on/off combinations)", FAC), ("Reference", REF), ("Manual ablations and controls (one change each, relative to the full model)", EXT)]
for gname, ms_ in groups:
    if not ms_: continue
    for st in ["mean", "median"]:
        num = head(PA, ms_, st); num.to_csv(TB / f"headline_{gname.split()[0].lower()}_{st}.csv")
        R_.append(f"## {gname}: Overall and Mean(6), {st}" + (" ± std" if st == "mean" else "") + "\n" + md(num, head(PA, ms_, "std") if st == "mean" else None) + "\n")
        print(f"\n=== {gname}: Overall and Mean(6) - {st} ===\n" + num.round(4).to_string())
for nm, sf in HEAD:
    for st in ["mean", "median"]:
        t = tbl(PA, FAC + REF + EXT, st, sf); t.to_csv(TB / f"perscore_{nm}_{st}.csv"); R_.append(f"## Per-score {nm}, all configurations, {st}\n" + md(t) + "\n")
R_.append("## Same with the mapping fitted on the test predictions (paper protocol), mean\n" + md(head(PA, FAC + REF + EXT, "mean", [("SRCC", "srcc"), ("PLCC", "plcc_test"), ("KRCC", "krcc"), ("RMSE", "rmse_test")])) + "\n")

# ---------- every configuration vs the full model (paired, same splits) ----------
pt = []
for m in [n for n in FAC + EXT + REF if n != FULL]:
    for nm, c_ in [("Overall SRCC", "O_srcc"), ("Mean(6) SRCC", "mean_srcc"), ("Overall PLCC (val-fit)", "O_plcc_val"), ("Overall RMSE (val-fit)", "O_rmse_val")]:
        a, b = PA[PA.model == m].set_index("seed")[c_].loc[seeds_A].values, PA[PA.model == FULL].set_index("seed")[c_].loc[seeds_A].values; d = a - b
        pt.append(dict(config=lab(m), group=GROUP[m], metric=nm, mean_diff_vs_full=d.mean(), median_diff=float(np.median(d)), seeds_worse_than_full=int((d > 0).sum() if "RMSE" in nm else (d < 0).sum()), n=len(d), p_wilcoxon=wp(a, b)))
PT = pd.DataFrame(pt)
if len(PT): PT["p_holm_within_metric"] = PT.groupby("metric")["p_wilcoxon"].transform(lambda s: holm(s.values))
PT.to_csv(TB / "paired_tests_vs_full.csv", index=False)
print("\nPaired tests vs the full model (negative mean_diff = the ablated model is worse; RMSE: positive = worse):\n" + PT.round(4).to_string(index=False)); R_.append("## Every configuration vs the full model (paired on the same splits; Holm correction within each metric)\n```\n" + PT.round(4).to_string(index=False) + "\n```\n")
DS = pd.DataFrame({SN[s]: [col(PA, m, f"{s}_srcc").mean() - col(PA, FULL, f"{s}_srcc").mean() for m in FAC + EXT + REF if m != FULL] for s in SC}, index=[lab(m) for m in FAC + EXT + REF if m != FULL])
DS.to_csv(TB / "srcc_delta_vs_full_per_score.csv"); print("\nMean SRCC change vs the full model, per score:\n" + DS.round(4).to_string()); R_.append("## Mean SRCC change vs the full model, per score\n```\n" + DS.round(4).to_string() + "\n```\n")

# ---------- factorial effects: the contrasts of the 2^k design (main effects, 2-way, 3-way, ... interactions) ----------
EFF = pd.DataFrame()
if len(FAC) == 2 ** len(FACTORS_ACTIVE):
    k = len(FACTORS_ACTIVE); bits_of = {n: tuple(int(n[2 * i + 1]) for i in range(k)) for n in FAC}; rows_e = []
    for metric, mname in [("O_srcc", "Overall SRCC"), ("mean_srcc", "Mean(6) SRCC"), ("O_rmse_val", "Overall RMSE (val-fit)")]:
        val = {s: {n: float(PA[(PA.model == n) & (PA.seed == s)][metric].iloc[0]) for n in FAC} for s in seeds_A}
        for r_ in range(1, k + 1):
            for U in itertools.combinations(range(k), r_):
                eff = np.array([sum(np.prod([2 * bits_of[n][i] - 1 for i in U]) * val[s][n] for n in FAC) / 2 ** (k - 1) for s in seeds_A])
                rows_e.append(dict(metric=mname, effect="x".join(FACTORS_ACTIVE[i] for i in U), order={1: "main", 2: "2-way", 3: "3-way", 4: "4-way", 5: "5-way"}[r_], mean_effect=eff.mean(), std=eff.std(ddof=1), seeds_positive=int((eff > 0).sum()), n=len(eff), p_wilcoxon=wp(eff)))
    EFF = pd.DataFrame(rows_e); EFF["p_holm_within_metric"] = EFF.groupby("metric")["p_wilcoxon"].transform(lambda s: holm(s.values)); EFF.to_csv(TB / "factorial_effects.csv", index=False)
    print("\nFactorial effects (main effect of X = mean over the other factors of [X present - X removed]; positive = X helps; interactions: x = joint effect beyond the sum of the parts):\n" + EFF.round(4).to_string(index=False))
    R_.append("## Factorial effects (2^k design)\nMain effect of X = average, over all combinations of the other factors, of (X present minus X removed); for RMSE negative = X helps. An interaction A x B is positive when A and B help more together than the sum of their separate effects.\n```\n" + EFF.round(4).to_string(index=False) + "\n```\n")
    best_cell = head(PA, FAC, "mean")["Overall SRCC"].idxmax(); R_.append(f"Best factorial cell by mean Overall SRCC: {best_cell}\n")

# ---------- manual ablations with a natural axis ----------
AX = {}
g_names = [("7x7 = 49 patches (full)", FULL), ("4x4", "A6_grid4"), ("3x3", "A6_grid3"), ("2x2", "A6_grid2")]; t_names = [("T=16 (full)", FULL), ("T=8", "A5_T8")]
for title, lst in [("patch grid (proxy for patch size, manual S29)", g_names), ("number of frames T (manual S30)", t_names)]:
    if all(n in set(PA.model) for _, n in lst):
        AX[title] = pd.DataFrame({"config": [a for a, _ in lst], **{f"{SN[s]} SRCC": [col(PA, n, f"{s}_srcc").mean() for _, n in lst] for s in ("O", "mean")}, "train s / run": [col(PA, n, "train_time_s").mean() for _, n in lst], "test ms / clip": [col(PA, n, "test_ms_per_clip").mean() for _, n in lst]})
        print(f"\n{title}:\n" + AX[title].round(4).to_string(index=False)); R_.append(f"## {title}\n```\n" + AX[title].round(4).to_string(index=False) + "\n```\n")
cons = [("none (final model)", FULL), ("0.05", "A9_consistency_0.05"), ("0.5", "A9_consistency_0.5")]
if all(n in set(PA.model) for _, n in cons):
    AX["lambda_t"] = pd.DataFrame({"lambda_t (added consistency loss)": [a for a, _ in cons], **{f"{SN[s]} SRCC": [col(PA, n, f"{s}_srcc").mean() for _, n in cons] for s in ("O", "mean")}, "Overall RMSE": [col(PA, n, "O_rmse_val").mean() for _, n in cons]})
    print("\nAdded temporal-consistency loss:\n" + AX["lambda_t"].round(4).to_string(index=False)); R_.append("## Added temporal-consistency loss\n```\n" + AX["lambda_t"].round(4).to_string(index=False) + "\n```\n")
lam = [("0 (no ranking)", "A9_rank0"), ("0.05", "A9_rank0.05"), ("0.1 (default)", FULL), ("0.2", "A9_rank0.2"), ("0.5", "A9_rank0.5")]
if all(n in set(PA.model) for _, n in lam):
    AX["lambda_r"] = pd.DataFrame({"lambda_r": [a for a, _ in lam], **{f"{SN[s]} SRCC": [col(PA, n, f"{s}_srcc").mean() for _, n in lam] for s in ("O", "mean")}, "Overall RMSE": [col(PA, n, "O_rmse_val").mean() for _, n in lam]})
    print("\nRanking-loss weight (manual S20):\n" + AX["lambda_r"].round(4).to_string(index=False)); R_.append("## Ranking-loss weight\n```\n" + AX["lambda_r"].round(4).to_string(index=False) + "\n```\n")

# ---------- attention monitors (manual S33, S39.3, S39.4) and the frame-order check ----------
AT = pd.DataFrame({lab(m): dict(spatial_entropy=col(PA, m, "ent_alpha").mean(), temporal_entropy=col(PA, m, "ent_beta").mean(), beta_attr_diff=col(PA, m, "beta_attr_diff").mean()) for m in FAC + EXT if PA[PA.model == m]["ent_alpha"].notna().any() or PA[PA.model == m]["ent_beta"].notna().any()}).T
AT.to_csv(TB / "attention_monitors.csv"); print("\nAttention monitors (entropy normalised to [0,1]: 1 = uniform, near 0 = collapsed; beta_attr_diff = how different the five attribute-specific temporal attentions are, 0 = identical):\n" + AT.round(4).to_string())
R_.append("## Attention monitors (collapse check, attribute specificity)\nEntropy is normalised to [0,1] (1 = uniform attention, near 0 = collapsed on one patch/frame). beta_attr_diff = mean pairwise half-L1 distance between the five attribute-specific beta curves (0 = identical, 1 = disjoint).\n```\n" + AT.round(4).to_string() + "\n```\n")
if "shuf_O_srcc" in PA.columns and PA[PA.model == FULL]["shuf_O_srcc"].notna().any():
    f_ = PA[PA.model == FULL].set_index("seed").loc[seeds_A]
    print(f"\nFull model with test frames shuffled (no retraining): Overall SRCC {f_['O_srcc'].mean():.4f} -> {f_['shuf_O_srcc'].mean():.4f} | Mean(6) {f_['mean_srcc'].mean():.4f} -> {f_['shuf_mean_srcc'].mean():.4f}  (a small drop means the model barely uses frame order)")
    R_.append(f"## Frame-order check (full model, test frames shuffled, not retrained)\nOverall SRCC {f_['O_srcc'].mean():.4f} -> {f_['shuf_O_srcc'].mean():.4f}; Mean(6) SRCC {f_['mean_srcc'].mean():.4f} -> {f_['shuf_mean_srcc'].mean():.4f}. A small drop means the model uses little temporal order.\n")

# ---------- Protocol B: patient-independent split ----------
if len(PB):
    mB = [m for m in PROTO_B_MODELS if m in set(PB.model)]; sB = sorted(set.intersection(*[set(PB.loc[PB.model == m, "seed"]) for m in mB]))
    PBt = PB[PB.model.isin(mB) & PB.seed.isin(sB)]; PAt = PA[PA.model.isin(mB) & PA.seed.isin(sB)]
    rowsB = [{"config": lab(m), **{f"{SN[s]} SRCC A (random)": col(PAt, m, f"{s}_srcc").mean() for s in ("O", "mean")}, **{f"{SN[s]} SRCC B (patient-indep.)": col(PBt, m, f"{s}_srcc").mean() for s in ("O", "mean")},
              "Overall drop": col(PAt, m, "O_srcc").mean() - col(PBt, m, "O_srcc").mean(), "Mean(6) drop": col(PAt, m, "mean_srcc").mean() - col(PBt, m, "mean_srcc").mean(), "Overall RMSE A": col(PAt, m, "O_rmse_val").mean(), "Overall RMSE B": col(PBt, m, "O_rmse_val").mean()} for m in mB]
    PBT = pd.DataFrame(rowsB).set_index("config"); PBT.to_csv(TB / "protocol_A_vs_B.csv")
    print(f"\nProtocol A (random) vs Protocol B (patient-independent), seeds {sB}; test-set size B ~ {int(PBt['n_test'].mean())} clips:\n" + PBT.round(4).to_string()); R_.append(f"## Protocol A vs Protocol B (patient-independent split), seeds {sB}\nCheck that PATIENT_REGEX really identifies the patient before quoting Protocol B (see Cell 2). Test-set size under B varies by seed (about {int(PBt['n_test'].mean())} clips).\n```\n" + PBT.round(4).to_string() + "\n```\n")

# ---------- DARNet, timing, parameters, hyper-parameters, logistic parameters ----------
val_rank = pd.Series({m: PA[PA.model == m]["val_mean_srcc"].mean() for m in FAC + EXT}).sort_values(ascending=False); BEST_CFG = val_rank.index[0]
def darnet_tab(m, mets=(("SRCC", "srcc"), ("PLCC", "plcc_test"), ("KRCC", "krcc"), ("RMSE", "rmse_test"))):
    rows_ = []
    for s in SC:
        row = {"Score": SN[s]}
        for i, (nm, sf) in enumerate(mets): v = col(PA, m, f"{s}_{sf}").mean(); row[f"{nm} DARNet"] = f"{DARNET[s][i]:.3f}"; row[f"{nm} ours"] = f"{v:.3f}"; row[f"{nm} gap"] = f"{v - DARNET[s][i]:+.3f}"
        rows_.append(row)
    return pd.DataFrame(rows_).set_index("Score")
for m in dict.fromkeys([FULL, BEST_CFG]):
    t = darnet_tab(m); t.to_csv(TB / f"vs_darnet_{m}.csv"); print(f"\n{lab(m)} vs DARNet (test-fit mapping as in the paper; RMSE gap > 0 is worse):\n" + t.to_string()); R_.append(f"## {lab(m)} vs DARNet (paper protocol)\n```\n" + t.to_string() + "\n```\n")
R_.append(f"Best configuration by mean validation Mean(6) SRCC (selected without test labels): {lab(BEST_CFG)} ({val_rank[BEST_CFG]:.4f}). Full model: {val_rank.get(FULL, float('nan')):.4f}.\n")
TIM = pd.DataFrame({lab(m): dict(params=PARAMS[m], epochs_run=col(PA, m, "epochs_run").mean(), best_epoch=col(PA, m, "best_epoch").mean(), train_s=col(PA, m, "train_time_s").mean(), test_ms_per_clip=col(PA, m, "test_ms_per_clip").mean()) for m in FAC + REF + EXT}).T
TIM.to_csv(TB / "timing_and_params.csv"); print("\nSize and time:\n" + TIM.round(3).to_string()); R_.append("## Size and time\n```\n" + TIM.round(3).to_string() + "\n```\n")
HPT = pd.Series({**HP, "epochs_max": EPOCHS, "batch": BATCH, "early_stop_patience": PATIENCE}); HPT.to_csv(TB / "hyperparameters.csv", header=["value"]); print("\nFixed hyper-parameters (identical for every configuration):\n" + HPT.to_string()); R_.append("## Fixed hyper-parameters (identical for every configuration)\n```\n" + HPT.to_string() + "\n```\n")
LP = pd.DataFrame([dict(config=lab(m), score=s, **{f"val-fit b{q}": col(PA, m, f"{s}_vb{q}").mean() for q in range(1, 5)}, **{f"test-fit b{q}": col(PA, m, f"{s}_tb{q}").mean() for q in range(1, 5)}) for m in [FULL] for s in TARGETS])
LP.to_csv(TB / "logistic_parameters_full_model.csv", index=False); print("\nLogistic-mapping parameters of the full model (mean over seeds):\n" + LP.round(3).to_string(index=False)); R_.append("## Logistic-mapping parameters, full model (mean over seeds)\n```\n" + LP.round(3).to_string(index=False) + "\n```\n")
(OUT / "README_ablation.md").write_text("\n".join(R_)); print(f"\nSaved tables to {TB} and README_ablation.md")


In [ ]:
# ============ CELL 9 — figures ============
BL, OR, IK, MU, GR, GN = "#2a78d6", "#eb6834", "#2b2b2a", "#8a8984", "#e4e3df", "#2e9e5b"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GR, "axes.axisbelow": True, "savefig.bbox": "tight", "legend.frameon": False})
FIG = OUT / "figures"; show = lambda: plt.show() if "ipykernel" in sys.modules else None
def save(fig, name): fig.savefig(FIG / name, dpi=300); show(); plt.close(fig)

# 1) all factorial cells, sorted
if FAC:
    order_ = sorted(FAC, key=lambda m: col(PA, m, "O_srcc").mean()); fig, ax = plt.subplots(1, 2, figsize=(11, 0.32 * len(order_) + 1.6))
    for a, (sc, xl, ref) in zip(ax, [("O", "Overall SRCC", DARNET["O"][0]), ("mean", "Mean(6) SRCC", DARNET["mean"][0])]):
        mu = [col(PA, m, f"{sc}_srcc").mean() for m in order_]; sd = [col(PA, m, f"{sc}_srcc").std() for m in order_]
        a.errorbar(mu, range(len(order_)), xerr=sd, fmt="o", color=BL, capsize=2); [a.plot(mu[i], i, "o", color=OR) for i, m in enumerate(order_) if m == FULL]
        a.axvline(ref, color=IK, ls="--", lw=1); a.annotate(f"DARNet {ref:.3f}", (ref, len(order_) - 0.4), ha="left", fontsize=8); a.set_yticks(range(len(order_))); a.set_yticklabels([lab(m) for m in order_] if a is ax[0] else []); a.set_xlabel(xl)
    fig.suptitle(f"All factorial cells, mean ± sd over {len(seeds_A)} seeds (orange = full model)", fontsize=9, x=0.01, ha="left"); save(fig, "ab_factorial_cells.png")

# 2) factorial effects
if len(EFF):
    fig, ax = plt.subplots(1, 2, figsize=(11, 0.3 * (2 ** len(FACTORS_ACTIVE)) + 1.2))
    for a, mname in zip(ax, ["Overall SRCC", "Mean(6) SRCC"]):
        e = EFF[EFF.metric == mname].reset_index(drop=True); cols_ = [OR if o == "main" else (BL if o == "2-way" else MU) for o in e["order"]]
        a.barh(range(len(e)), e["mean_effect"], xerr=e["std"] / np.sqrt(e["n"]), color=cols_, error_kw=dict(ecolor=IK, lw=0.8)); a.axvline(0, color=IK, lw=0.8)
        a.set_yticks(range(len(e))); a.set_yticklabels(e["effect"] if a is ax[0] else []); a.invert_yaxis(); a.set_xlabel(f"effect on {mname}  (± s.e.)")
    ax[0].set_title("orange: main effects | blue: 2-way | grey: higher-order interactions", loc="left", fontsize=8); save(fig, "ab_factorial_effects.png")

# 3) 2D layout of the 4-factor design (rows S,T ; columns M,P)
if len(FAC) == 16 and all(f in FACTORS_ACTIVE for f in "STMP"):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    for a, (sc, ttl) in zip(ax, [("O", "Overall SRCC"), ("mean", "Mean(6) SRCC")]):
        M_ = np.zeros((4, 4)); rows_ = [(s, t) for s in (1, 0) for t in (1, 0)]; cols2 = [(m, p) for m in (1, 0) for p in (1, 0)]
        for i, (s_, t_) in enumerate(rows_):
            for j, (m_, p_) in enumerate(cols2):
                nm = next(n for n in FAC if n[:8] == f"S{s_}T{t_}M{m_}P{p_}"); M_[i, j] = col(PA, nm, f"{sc}_srcc").mean()
        im = a.imshow(M_, cmap="Blues"); a.grid(False); a.set_xticks(range(4)); a.set_xticklabels([f"M{m}P{p}" for m, p in cols2]); a.set_yticks(range(4)); a.set_yticklabels([f"S{s}T{t}" for s, t in rows_]); a.set_title(ttl, loc="left")
        for i in range(4):
            for j in range(4): a.text(j, i, f"{M_[i, j]:.4f}", ha="center", va="center", fontsize=8, color="white" if M_[i, j] > (M_.max() + M_.min()) / 2 else IK)
    save(fig, "ab_factorial_grid.png")

# 4) manual ablations / controls: change vs the full model
if EXT:
    fig, ax = plt.subplots(1, 2, figsize=(11, 0.34 * len(EXT) + 1.4))
    for a, (sc, xl) in zip(ax, [("O", "Overall SRCC change vs full model"), ("mean", "Mean(6) SRCC change vs full model")]):
        full_v = PA[PA.model == FULL].set_index("seed")[f"{sc}_srcc"].loc[seeds_A].values
        d = [PA[PA.model == m].set_index("seed")[f"{sc}_srcc"].loc[seeds_A].values - full_v for m in EXT]
        a.errorbar([x.mean() for x in d], range(len(EXT)), xerr=[x.std(ddof=1) / np.sqrt(len(x)) for x in d], fmt="o", color=BL, capsize=2); a.axvline(0, color=IK, lw=0.8)
        a.set_yticks(range(len(EXT))); a.set_yticklabels(EXT if a is ax[0] else []); a.invert_yaxis(); a.set_xlabel(xl + "  (± s.e.)")
    save(fig, "ab_manual_ablations_vs_full.png")

# 5) one-axis plots: patch grid, T, lambda_r
axes_ = [k for k in ("patch grid (proxy for patch size, manual S29)", "number of frames T (manual S30)", "lambda_r") if k in AX]
if axes_:
    fig, ax = plt.subplots(1, len(axes_), figsize=(4.2 * len(axes_), 3.2), squeeze=False)
    for a, k in zip(ax[0], axes_):
        t = AX[k]; a.plot(range(len(t)), t["Overall SRCC"], "o-", color=BL, label="Overall"); a.plot(range(len(t)), t["Mean(6) SRCC"], "s-", color=OR, label="Mean(6)")
        a.set_xticks(range(len(t))); a.set_xticklabels(t.iloc[:, 0], rotation=20, ha="right"); a.set_ylabel("SRCC"); a.set_title(k, loc="left", fontsize=8)
    ax[0][0].legend(); save(fig, "ab_axis_plots.png")

# 6) attention monitors
if len(AT):
    fig, ax = plt.subplots(1, 3, figsize=(12, 0.3 * len(AT) + 1.4))
    for a, (c, ttl) in zip(ax, [("spatial_entropy", "spatial attention entropy"), ("temporal_entropy", "temporal attention entropy"), ("beta_attr_diff", "attribute-specific beta difference")]):
        a.barh(range(len(AT)), AT[c].fillna(0), color=BL); a.set_yticks(range(len(AT))); a.set_yticklabels(AT.index if a is ax[0] else []); a.invert_yaxis(); a.set_title(ttl, loc="left", fontsize=8)
    save(fig, "ab_attention_monitors.png")

# 7) Protocol A vs B
if len(PB) and "PBT" in globals():
    fig, ax = plt.subplots(figsize=(7, 3.4)); x = np.arange(len(PBT)); w = 0.38
    ax.bar(x - w / 2, PBT["Mean(6) SRCC A (random)"], w, color=BL, label="A: random split"); ax.bar(x + w / 2, PBT["Mean(6) SRCC B (patient-indep.)"], w, color=OR, label="B: patient-independent")
    ax.set_xticks(x); ax.set_xticklabels(PBT.index, rotation=20, ha="right"); ax.set_ylabel("Mean(6) SRCC"); lo_ = min(PBT["Mean(6) SRCC B (patient-indep.)"].min(), PBT["Mean(6) SRCC A (random)"].min()); ax.set_ylim(max(0, lo_ - 0.05), None); ax.legend(); save(fig, "ab_protocol_A_vs_B.png")

# 8) per-score SRCC change vs the full model
if len(DS):
    gap = DS.values; lim = max(0.01, np.abs(gap).max()); fig, ax = plt.subplots(figsize=(7.5, 0.34 * len(DS) + 1.4)); ax.imshow(gap, cmap="RdBu", aspect="auto", vmin=-lim, vmax=lim); ax.grid(False)
    ax.set_xticks(range(len(SC))); ax.set_xticklabels(list(DS.columns)); ax.set_yticks(range(len(DS))); ax.set_yticklabels(DS.index, fontsize=7)
    for i in range(gap.shape[0]):
        for j in range(gap.shape[1]): ax.text(j, i, f"{gap[i, j]:+.3f}", ha="center", va="center", fontsize=7, color="white" if abs(gap[i, j]) > 0.6 * lim else IK)
    ax.set_title("Mean SRCC change vs the full model (blue: better, red: worse)", loc="left"); save(fig, "ab_srcc_change_per_score.png")
print("Figures saved to", FIG)


In [ ]:
# ============ CELL 10 — zip everything ============
ZIP = WORK / f"cvstqa_ablation2_set{SET_ID}_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(OUT.parent))
print(f"Wrote {ZIP} ({ZIP.stat().st_size / 1e6:.1f} MB): runs.csv, tables, figures, per-run predictions, README_ablation.md")
